In [1]:
##### GPU is required to run this kernel (Ran on Kaggle GPU originally)

In [2]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import *
from sklearn.preprocessing import *
from sklearn.metrics import *
from lightgbm import LGBMClassifier
import gc

from tqdm import tqdm, tqdm_notebook

import warnings
warnings.simplefilter('ignore')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import datetime
import time

In [3]:
# import multiprocessing as mp
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.optimizers import *
from tensorflow.keras import regularizers
from tensorflow.keras.activations import *
import tensorflow as tf

In [5]:
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
ss = pd.read_csv('SampleSubmission.csv')

In [6]:
prod_cols = ['P5DA', 'RIBP', '8NN1','7POT', '66FJ', 'GYSR', 'SOP4', 'RVSZ', 'PYUQ', 'LJR9', 'N2MW', 'AHXO',
             'BSTQ', 'FM3X', 'K6QO', 'QBOL', 'JWFN', 'JZ9D', 'J9JW', 'GHYX', 'ECY3']

In [7]:
ID_COL, TARGET_COL = 'ID', 'target'

In [8]:
miss_prod_lst = []
for p in prod_cols:
    tmp_df = train[train[p] == 1]
    tmp_df['target'] = p
    tmp_df[p] = 0  
    miss_prod_lst.append(tmp_df)
train = pd.concat(miss_prod_lst, axis=0)

In [9]:
train = train.sample(frac=1, random_state=2)
mapper = pd.Series(index = train['target'].unique(), data = np.arange(train['target'].nunique())).to_dict()
rev_mapper = {val: key for key, val in mapper.items()}
train['target'] = train['target'].map(mapper)

In [10]:
df = pd.concat([train, test]).reset_index(drop = True)
D_COL = 'join_date'
df[D_COL] = pd.to_datetime(df[D_COL], format = '%d/%m/%Y')
for f in ['month', 'year', 'week']:
    df[f] = getattr(df[D_COL].dt, f)
cat_cols = ['marital_status', 'branch_code', 'occupation_code', 'occupation_category_code', 'sex']
df[cat_cols] = df[cat_cols].apply(lambda x: pd.factorize(x)[0])
df['total_products_recommended'] = df[prod_cols].sum(axis = 1) # + 1
df['day_since_start'] = (df[D_COL] - df[D_COL].min()).dt.days

prod_cols_freq_sorted = df['target'].map(rev_mapper).value_counts().index.tolist()
prod_df = df[prod_cols].copy().astype('str')

In [11]:
for c in ['occupation_code', 'branch_code', 'birth_year', 'total_products_recommended', 'join_date']:
    df[f'n_unq_customers_in_{c}'] = df.groupby(c)['ID'].transform('nunique')

In [12]:
for p in tqdm_notebook(prod_cols_freq_sorted):
    df[f'branch_code_{p}_sum'] = df['branch_code'].map(df.groupby('branch_code')[p].sum())

In [13]:
for p in tqdm_notebook(prod_cols_freq_sorted):
    fltrd_data = df[df[p] == 1]
    df[f'occupation_code_{p}_nunq_customers'] = df['occupation_code'].map(fltrd_data.groupby('occupation_code')["ID"].nunique())
    df[f'branch_code_{p}_nunq_customers'] = df['branch_code'].map(fltrd_data.groupby('branch_code')["ID"].nunique())

In [14]:
df['branch_occupation_nunique'] = df['branch_code'].map(df.groupby('branch_code')['occupation_code'].nunique())
df['occupation_branch_nunique'] = df['occupation_code'].map(df.groupby('occupation_code')['branch_code'].nunique())

In [15]:
from itertools import combinations

cc = list(combinations(prod_cols,2))
tmp = pd.concat([df[c[1]].sub(df[c[0]]) for c in cc], axis=1, keys=cc)
tmp.columns = tmp.columns.map(''.join)
print (tmp.shape)
df = pd.concat([df, tmp], axis=1)

(76353, 210)


In [16]:
df.loc[~df['marital_status'].isin([0,1,2]), 'marital_status'] = np.nan


In [17]:
df['prod_comb'] = ''
for p in tqdm_notebook(prod_cols_freq_sorted):
    df['prod_comb'] = df['prod_comb'] + df[p].astype('str')
df['prod_comb'] = pd.factorize(df['prod_comb'])[0]

In [18]:
df['prod_comb_freq'] = df['prod_comb'].map(df['prod_comb'].value_counts(normalize=True))

In [19]:
vc = df['prod_comb'].value_counts()
less_freq_prod_comb = vc[vc < 5].index.tolist()
df.loc[df['prod_comb'].isin(less_freq_prod_comb), 'prod_comb'] = np.nan

In [20]:
df['join_date'] = pd.factorize(df['join_date'])[0]

In [21]:
features = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]
cat_cols = ['join_date','sex','marital_status','birth_year','branch_code','occupation_code','occupation_category_code', 'prod_comb', 'month', 'year', 'week']
num_cols = [c for  c in features if c not in cat_cols]
df[cat_cols] = df[cat_cols].apply(lambda x: x.fillna(-999))

In [22]:
def clear_inf(x):
  x[(x == np.inf) | (x == -np.inf)] = np.nan
  return x
df[num_cols] = df[num_cols] = df[num_cols].apply(lambda x: clear_inf(x))
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.mean() + 1e-6))
df[[c + '_log' for c in num_cols]] = df[num_cols].apply(lambda x: np.log1p(x))
num_cols = num_cols + [c + '_log' for c in num_cols]
df[num_cols] = df[num_cols] = df[num_cols].apply(lambda x: clear_inf(x))
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.mean() + 1e-6))

scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
df[cat_cols] = df[cat_cols].astype('int')
df = pd.concat([df, pd.get_dummies(df[cat_cols], columns = cat_cols)], axis=1)

In [23]:
features = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]
num_cols = [c for  c in features if c not in cat_cols]
len(num_cols)

1402

In [24]:
train, test = df[:train.shape[0]].reset_index(drop = True), df[train.shape[0]:].reset_index(drop = True)
train.shape, test.shape

((66353, 1415), (10000, 1415))

In [25]:
tmp = train[train['target'] == 20]
train = pd.concat([train, tmp], axis=0)

In [26]:
N_CLASSES = train['target'].nunique()
N_CLASSES

21

In [27]:
def build_embedding_network(categorical_vars, num_cols, y_nunq, nunique_dict, max_embed_size=50):
    embeddings = []
    inputs = []

    for categorical_var in categorical_vars:    
        single_input = Input(shape=(1,))
        no_of_unique_cat  = nunique_dict[categorical_var]
        embedding_size = min(np.ceil((no_of_unique_cat)/2), max_embed_size)
        embedding_size = max(2, embedding_size)
        embedding_size = int(embedding_size)
        embedding = Embedding(no_of_unique_cat, embedding_size, input_length=1)(single_input)
        embedding = Reshape(target_shape=(embedding_size,))(embedding)
        inputs.append(single_input)
        embeddings.append(embedding)

    input_numeric = Input(shape=(len(num_cols),))
    inputs.append(input_numeric)
    embeddings.append(input_numeric)


    x = concatenate(embeddings)
    for i in range(4):
        x = Dense(512)(x)
        x =  PReLU()(x)
        x = Dropout(0.24)(x)
    output = Dense(21, activation='softmax')(x)
    model = Model(inputs, output)
    opt = Nadam(lr=1e-4)
    model.compile(loss='categorical_crossentropy', optimizer=opt)
    
    return model

In [28]:
es = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                      patience = 15, 
                                      verbose=1, 
                                      restore_best_weights=True)

In [29]:
oofs = np.zeros((len(train), 21))
preds = np.zeros((len(test), 21))
fi_df = pd.DataFrame()

N_SPLITS = 5
folds = StratifiedKFold(N_SPLITS)

for fold_, (trn_idx, val_idx) in enumerate(folds.split(train, train["target"])):

  print(f'\n\n Fold {fold_} \n\n')
  X_trn, y_trn = train[features].iloc[trn_idx], train[TARGET_COL].iloc[trn_idx]
  X_val, y_val = train[features].iloc[val_idx], train[TARGET_COL].iloc[val_idx]
  X_tst = test[features]

  ### Adding categorical mappings
  nunique_dict = {c: X_trn[c].nunique() for c in cat_cols}
  cat_maps = {c: pd.Series(index = np.unique(X_trn[c]), data = np.arange(X_trn[c].nunique())) for c in cat_cols}
  proc_trn = [X_trn[c].map(cat_maps[c]).values for c in cat_cols]
  proc_val = [X_val[c].map(cat_maps[c]).values for c in cat_cols]
  proc_tst = [X_tst[c].map(cat_maps[c]).values for c in cat_cols]

  clf = build_embedding_network(cat_cols, num_cols, N_CLASSES, nunique_dict, 100)

  ### Adding numerical data
  proc_trn.append(X_trn[num_cols].values)
  proc_val.append(X_val[num_cols].values)
  proc_tst.append(X_tst[num_cols].values)

  gc.collect()

  print('\n\n ---- Starting NN Training ---- \n\n')
  history = clf.fit(proc_trn, pd.get_dummies(y_trn), batch_size=256, epochs=100, validation_data = (proc_val, pd.get_dummies(y_val)), callbacks=[es])

  oofs[val_idx] = clf.predict(proc_val)

  val_score = accuracy_score(y_val, oofs[val_idx].argmax(axis=1))
  print(f'\n\n Accuracy score for val is : {val_score}')

  preds += clf.predict(proc_tst) / N_SPLITS



 Fold 0 




0



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 3s 14ms/step - loss: 1.0169 - val_loss: 0.5690
Epoch 2/100
208/208 [==============================] - 2s 11ms/step - loss: 0.5806 - val_loss: 0.5022
Epoch 3/100
208/208 [==============================] - 3s 12ms/step - loss: 0.5161 - val_loss: 0.4469
Epoch 4/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4818 - val_loss: 0.4483
Epoch 5/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4574 - val_loss: 0.4242
Epoch 6/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4411 - val_loss: 0.4198
Epoch 7/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4289 - val_loss: 0.4062
Epoch 8/100
208/208 [==============================] - 3s 12ms/step - loss: 0.4194 - val_loss: 0.4019
Epoch 9/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4074 - val_loss: 0.4212
Epoch 10/100
208/208 [=======================

11788



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 3s 14ms/step - loss: 1.0163 - val_loss: 0.6030
Epoch 2/100
208/208 [==============================] - 2s 11ms/step - loss: 0.5751 - val_loss: 0.5042
Epoch 3/100
208/208 [==============================] - 2s 10ms/step - loss: 0.5088 - val_loss: 0.4714
Epoch 4/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4769 - val_loss: 0.4459
Epoch 5/100
208/208 [==============================] - 3s 13ms/step - loss: 0.4522 - val_loss: 0.4378
Epoch 6/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4367 - val_loss: 0.4388
Epoch 7/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4254 - val_loss: 0.4423
Epoch 8/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4148 - val_loss: 0.4148
Epoch 9/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4057 - val_loss: 0.4109
Epoch 10/100
208/208 [=======================

13556



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 3s 12ms/step - loss: 1.0142 - val_loss: 0.6169
Epoch 2/100
208/208 [==============================] - 2s 11ms/step - loss: 0.5803 - val_loss: 0.5008
Epoch 3/100
208/208 [==============================] - 3s 16ms/step - loss: 0.5159 - val_loss: 0.4640
Epoch 4/100
208/208 [==============================] - 2s 12ms/step - loss: 0.4799 - val_loss: 0.4403
Epoch 5/100
208/208 [==============================] - 3s 13ms/step - loss: 0.4582 - val_loss: 0.4418
Epoch 6/100
208/208 [==============================] - 2s 12ms/step - loss: 0.4433 - val_loss: 0.4219
Epoch 7/100
208/208 [==============================] - 3s 12ms/step - loss: 0.4309 - val_loss: 0.4279
Epoch 8/100
208/208 [==============================] - 3s 12ms/step - loss: 0.4157 - val_loss: 0.4111
Epoch 9/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4086 - val_loss: 0.4076
Epoch 10/100
208/208 [=======================

13661



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 3s 12ms/step - loss: 1.0208 - val_loss: 0.6155
Epoch 2/100
208/208 [==============================] - 3s 13ms/step - loss: 0.5780 - val_loss: 0.5382
Epoch 3/100
208/208 [==============================] - 3s 12ms/step - loss: 0.5123 - val_loss: 0.4806
Epoch 4/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4756 - val_loss: 0.4640
Epoch 5/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4541 - val_loss: 0.4619
Epoch 6/100
208/208 [==============================] - 3s 12ms/step - loss: 0.4379 - val_loss: 0.4417
Epoch 7/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4230 - val_loss: 0.4189
Epoch 8/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4149 - val_loss: 0.4216
Epoch 9/100
208/208 [==============================] - 3s 14ms/step - loss: 0.4035 - val_loss: 0.4226
Epoch 10/100
208/208 [=======================

13661



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 3s 13ms/step - loss: 0.9986 - val_loss: 0.5788
Epoch 2/100
208/208 [==============================] - 2s 12ms/step - loss: 0.5785 - val_loss: 0.5011
Epoch 3/100
208/208 [==============================] - 2s 12ms/step - loss: 0.5127 - val_loss: 0.4586
Epoch 4/100
208/208 [==============================] - 3s 12ms/step - loss: 0.4796 - val_loss: 0.4419
Epoch 5/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4575 - val_loss: 0.4303
Epoch 6/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4408 - val_loss: 0.4216
Epoch 7/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4277 - val_loss: 0.4202
Epoch 8/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4191 - val_loss: 0.4098
Epoch 9/100
208/208 [==============================] - 3s 13ms/step - loss: 0.4068 - val_loss: 0.4004
Epoch 10/100
208/208 [=======================

In [30]:
print(f'\nOOF val score: {log_loss(train[TARGET_COL], oofs)}')


OOF val score: 0.3877179195873387


In [31]:
oofs_df = pd.DataFrame(oofs).rename(rev_mapper, axis=1)
oofs_df['ID'] = train['ID'].values
oofs_df_melted = pd.melt(oofs_df, id_vars = 'ID', value_vars = prod_cols, var_name='prod', value_name = 'target')

In [32]:
train_cpy = train[['ID', 'target']]
train_cpy['target'] = train_cpy['target'].map(rev_mapper)
train_cpy = pd.get_dummies(train_cpy, columns = ['target'], prefix='', prefix_sep='')
train_cpy = train_cpy[oofs_df.columns.tolist()]
train_cpy_melted = pd.melt(train_cpy, id_vars = 'ID', value_vars = prod_cols, var_name='prod', value_name = 'target')
oofs_df_melted.loc[train_cpy_melted['target'] == 1, 'target'] = 1
log_loss(train_cpy_melted['target'], oofs_df_melted['target'])

0.011615940977742914

In [33]:
test_preds_df = pd.DataFrame(preds).rename(rev_mapper, axis=1)
test_preds_df['ID'] = test['ID'].values
test_preds_melted = pd.melt(test_preds_df, id_vars = 'ID', value_vars = prod_cols, var_name='prod', value_name = 'target')

In [34]:
test_preds_melted['ID X PCODE'] = test_preds_melted['ID'].astype('str') + ' X ' + test_preds_melted['prod']
mapper = pd.Series(index = test_preds_melted['ID X PCODE'], data = test_preds_melted['target'].values)

In [35]:
fltr = ss['Label'] != 1
ss.loc[fltr, 'Label'] = ss.loc[fltr, 'ID X PCODE'].map(mapper)
ss['Label'].describe().round(4)
ss.to_csv('m14_v2.csv', index=False)

count    210000.0000
mean          0.1088
std           0.2999
min           0.0000
25%           0.0000
50%           0.0000
75%           0.0017
max           1.0000
Name: Label, dtype: float64

In [36]:
ss.head(10)

,ID X PCODE,Label
0,F86J5PC X P5DA,1.047431e-05
1,F86J5PC X RIBP,5.031278e-05
2,F86J5PC X 8NN1,1.415795e-06
3,F86J5PC X 7POT,9.152110e-06
4,F86J5PC X 66FJ,1.951998e-05
5,F86J5PC X GYSR,3.429649e-07
6,F86J5PC X SOP4,2.831553e-05
7,F86J5PC X RVSZ,1.000000e+00
8,F86J5PC X PYUQ,3.877137e-04
9,F86J5PC X LJR9,1.488498e-05
